In [0]:
%run ./01-config

In [0]:
class HistoryLoader():
    def __init__(self, env):
        conf = Config()
        self.landing_zone = conf.base_dir_data + "/raw"
        self.test_data_dir = conf.base_dir_data + "/test_data"
        self.catalog = env
        self.db_name = conf.silver_db
        
    def load_date_lookup(self):
        print("Loading date_lookup table...", end='')
        spark.sql(f"""INSERT OVERWRITE TABLE {self.catalog}.{self.db_name}.date_lookup
                      SELECT date, week, year, month, dayofweek, dayofmonth, dayofyear, week_part 
                      FROM json.`{self.test_data_dir}/date_lookup.json/}`""")
    
    def load_with_timespan(self):
        import time
        start = int(time.now())
        print("Loading history table...", end='')
        self.load_date_lookup()
        print(f"Historical data loaded in {int(time.now()) - start} seconds")
    
    def assert_count(self, table_name, expected_count):):
        print(f"Validating record counts in {table_name}...")
        actual_count = spark.read.table(f"{self.catalog}.{self.db_name}.{table_name}").count()
        assert actual_count == expected_count, f"Expected {expected_count} records in {table_name}, found {actual_count}"
        print(f"Validated record counts in {table_name}")
    
    def validate(self):
        import time
        start = int(time.now())
        print("\nStarting historical data validation...")
        self.assert_count("date_lookup", 365)
        print(f"Historical data validated in {int(time.now()) - start} seconds")